<a href="https://colab.research.google.com/github/miriamamin1213-ux/Ablation_studies/blob/main/HecktorGPT2Gen%20CI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# 1. IMPORT LIBRARIES, SET RANDOM SEED AND SELECT DEVICE
# ============================================================
import os,random,re,gdown
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import Dataset,DataLoader
from transformers import GPT2Tokenizer,GPT2LMHeadModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,balanced_accuracy_score,roc_auc_score,average_precision_score,confusion_matrix,f1_score,roc_curve,precision_recall_curve

SEED=42
def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"]=str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"]=":4096:8"
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True,warn_only=True)
    torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32=False
        torch.backends.cudnn.allow_tf32=False

seed_everything(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:",device)

# ============================================================
# 2. LOAD AND PREPARE HECKTOR DATA
# ============================================================
DATA_FILE="HPV2025.xlsx"
if not os.path.exists(DATA_FILE):
    path=gdown.download(id="1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv",output=DATA_FILE,quiet=False,fuzzy=True)
    if path is None or not os.path.exists(DATA_FILE):
        try:
            from google.colab import files
            uploaded=files.upload()
        except ImportError:
            raise FileNotFoundError("Download HPV2025.xlsx and place it beside the notebook.")
        if DATA_FILE not in uploaded:raise FileNotFoundError("Please upload HPV2025.xlsx.")

df=pd.read_excel(DATA_FILE)
required=["HPV Status","Age","Gender","Tobacco Consumption","Alcohol Consumption","Performance Status","T-stage","N-stage","M-stage"]
missing=[c for c in required if c not in df.columns]
if missing:raise KeyError(f"Missing columns: {missing}")
df=df.dropna(subset=["HPV Status"]).copy()
df=df.drop(columns=["PatientID","CenterID","Task 1","Task 2","Task 3","Relapse","RFS","Treatment"],errors="ignore")
df["HPV Status"]=pd.to_numeric(df["HPV Status"],errors="coerce")
df=df[df["HPV Status"].isin([0,1])].copy()
df["HPV Status"]=df["HPV Status"].astype(int)

def stage_num(x):
    if pd.isna(x):return np.nan
    if isinstance(x,str):
        s=x.strip().upper();match=re.fullmatch(r"[TNM]\s*(\d+)",s)
        if match:return int(match.group(1))
        try:return float(s)
        except ValueError:return np.nan
    try:return float(x)
    except (TypeError,ValueError):return np.nan

for c in ["T-stage","N-stage","M-stage"]:df[c]=df[c].apply(stage_num)
features=["Age","Gender","Tobacco Consumption","Alcohol Consumption","Performance Status","T-stage","N-stage","M-stage"]
df=df[features+["HPV Status"]].copy()
print("Patients:",len(df))
print("HPV class distribution:\n",df["HPV Status"].value_counts().sort_index())
print("Features:",features)

# ============================================================
# 3. STRATIFIED TRAIN/VALIDATION/TEST SPLIT (65/15/20)
# ============================================================
train_val_df,test_df=train_test_split(df,test_size=0.20,random_state=SEED,stratify=df["HPV Status"])
train_df,val_df=train_test_split(train_val_df,test_size=0.1875,random_state=SEED,stratify=train_val_df["HPV Status"])
train_df,val_df,test_df=(x.copy() for x in [train_df,val_df,test_df])

# ============================================================
# 4. HANDLE MISSING VALUES USING TRAINING DATA ONLY
# ============================================================
fill_values={}
for c in features:
    if pd.api.types.is_numeric_dtype(train_df[c]):
        value=train_df[c].median()
        value=0 if pd.isna(value) else value
    else:
        modes=train_df[c].dropna().astype(str).mode()
        value=modes.iloc[0] if not modes.empty else "Unknown"
    fill_values[c]=value
    for split in [train_df,val_df,test_df]:
        split[c]=split[c].fillna(value)
        if split[c].dtype==object:
            split[c]=split[c].astype(str).replace({"":"Unknown","nan":"Unknown","None":"Unknown"})

y_train=train_df["HPV Status"].to_numpy(dtype=int)
y_val=val_df["HPV Status"].to_numpy(dtype=int)
y_test=test_df["HPV Status"].to_numpy(dtype=int)
print("Train:",len(train_df),"classes:",np.bincount(y_train,minlength=2))
print("Validation:",len(val_df),"classes:",np.bincount(y_val,minlength=2))
print("Test:",len(test_df),"classes:",np.bincount(y_test,minlength=2))

# ============================================================
# 5. CONVERT HECKTOR FEATURES INTO CLINICAL TEXT PROMPTS
# ============================================================
def display_value(x):
    if pd.isna(x):return "Unknown"
    if isinstance(x,(float,np.floating)) and float(x).is_integer():return str(int(x))
    return str(x).strip()

def stage_value(prefix,x):
    value=display_value(x)
    if value=="Unknown":return value
    return value if value.upper().startswith(prefix) else f"{prefix}{value}"

def patient_to_text(row):
    gender_value=display_value(row["Gender"])
    tobacco_value=display_value(row["Tobacco Consumption"])
    alcohol_value=display_value(row["Alcohol Consumption"])
    gender={"1":"male","0":"female"}.get(gender_value.lower(),gender_value)
    smoker={"1":"current smoker","0":"non-smoker"}.get(tobacco_value.lower(),tobacco_value)
    alcohol={"1":"consumes alcohol","0":"does not consume alcohol"}.get(alcohol_value.lower(),alcohol_value)
    return ("Q: Predict this patient's Human Papillomavirus (HPV) status using the clinical information available. "
        f"Age at diagnosis: {display_value(row['Age'])}. Gender: {gender}. Tobacco consumption: {smoker}. "
        f"Alcohol consumption: {alcohol}. Performance status: {display_value(row['Performance Status'])}. "
        f"Tumour stage: {stage_value('T',row['T-stage'])}. Nodal stage: {stage_value('N',row['N-stage'])}. "
        f"Metastatic stage: {stage_value('M',row['M-stage'])}. Is this patient HPV positive? Answer with only Yes or No.\nA:")

train_texts=[patient_to_text(r) for _,r in train_df.iterrows()]
val_texts=[patient_to_text(r) for _,r in val_df.iterrows()]
test_texts=[patient_to_text(r) for _,r in test_df.iterrows()]
train_targets=["Yes" if y==1 else "No" for y in y_train]
val_targets=["Yes" if y==1 else "No" for y in y_val]
test_targets=["Yes" if y==1 else "No" for y in y_test]
print("Example clinical narrative:\n",train_texts[0],"\nTarget:",train_targets[0])

# ============================================================
# 6. LOAD TOKENIZER AND SET MAXIMUM SEQUENCE LENGTH
# ============================================================
MODEL_NAME="openai-community/gpt2"
tokenizer=GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token=tokenizer.eos_token
TRAIN_TARGET_LENGTHS=[len(tokenizer.encode(" "+t,add_special_tokens=False))+1 for t in train_targets]
TRAIN_PROMPT_LENGTHS=[len(tokenizer.encode(t,add_special_tokens=False)) for t in train_texts]
TRAIN_REQUIRED_LENGTHS=[p+t for p,t in zip(TRAIN_PROMPT_LENGTHS,TRAIN_TARGET_LENGTHS)]
print("\n===== TOKEN LENGTH CHECK =====")
print("Training prompt tokens: min =",min(TRAIN_PROMPT_LENGTHS),"| median =",int(np.median(TRAIN_PROMPT_LENGTHS)),"| 95th percentile =",int(np.percentile(TRAIN_PROMPT_LENGTHS,95)),"| max =",max(TRAIN_PROMPT_LENGTHS))
print("Training prompt + answer tokens: max =",max(TRAIN_REQUIRED_LENGTHS))
MAX_LENGTH=min(1024,max(32,int(np.ceil(max(TRAIN_REQUIRED_LENGTHS)/8)*8)))
if max(TRAIN_REQUIRED_LENGTHS)>1024:
    print("WARNING: Some examples exceed GPT-2's 1024-token context window and will be truncated.")
print("Selected MAX_LENGTH:",MAX_LENGTH)
BATCH_SIZE=64;EPOCHS=500;LEARNING_RATE=1e-3;WEIGHT_DECAY=1e-4
print("Batch size:",BATCH_SIZE,"| Epochs:",EPOCHS)

# ============================================================
# 7. BUILD TOKENIZED DATASETS AND DATA LOADERS
# ============================================================
class GPT2GenerationDataset(Dataset):
    def __init__(self,texts,targets,labels):
        self.items=[]
        for text,target,label in zip(texts,targets,labels):
            prompt_ids=tokenizer.encode(text,add_special_tokens=False)
            target_ids=tokenizer.encode(" "+target,add_special_tokens=False)+[tokenizer.eos_token_id]
            prompt_ids=prompt_ids[:max(0,MAX_LENGTH-len(target_ids))]
            ids=prompt_ids+target_ids
            loss_labels=[-100]*len(prompt_ids)+target_ids
            pad_len=MAX_LENGTH-len(ids)
            self.items.append((ids+[tokenizer.pad_token_id]*pad_len,[1]*len(ids)+[0]*pad_len,loss_labels+[-100]*pad_len,int(label)))
    def __len__(self):return len(self.items)
    def __getitem__(self,i):
        ids,mask,labels,label=self.items[i]
        return {"input_ids":torch.tensor(ids,dtype=torch.long),"attention_mask":torch.tensor(mask,dtype=torch.long),"labels":torch.tensor(labels,dtype=torch.long),"class_label":torch.tensor(label,dtype=torch.long)}

train_dataset=GPT2GenerationDataset(train_texts,train_targets,y_train)
val_dataset=GPT2GenerationDataset(val_texts,val_targets,y_val)
test_dataset=GPT2GenerationDataset(test_texts,test_targets,y_test)
train_generator=torch.Generator().manual_seed(SEED)
train_loader=DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,generator=train_generator,num_workers=0)
val_loader=DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0)

# ============================================================
# 8. INITIALISE GPT-2, CLASS WEIGHTS AND ADAMW OPTIMIZER
# ============================================================
model=GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)
model.config.pad_token_id=tokenizer.pad_token_id
print("GPT-2 hidden size:",model.config.n_embd,"| Layers:",model.config.n_layer,"| Heads:",model.config.n_head)
class_counts=np.bincount(y_train,minlength=2)
class_weights=torch.tensor(len(y_train)/(2*np.maximum(class_counts,1)),dtype=torch.float32,device=device)
optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
print("Class weights:",class_weights.detach().cpu().numpy())

# ============================================================
# 9. DEFINE CLASS-WEIGHTED GENERATION LOSS
# ============================================================
def weighted_generation_loss(logits,labels,class_labels):
    shift_logits=logits[:,:-1,:].contiguous()
    shift_labels=labels[:,1:].contiguous()
    token_loss=nn.CrossEntropyLoss(reduction="none")(shift_logits.reshape(-1,shift_logits.size(-1)),shift_labels.reshape(-1)).view(shift_labels.size(0),-1)
    valid=(shift_labels!=-100).float()
    example_loss=(token_loss*valid).sum(1)/valid.sum(1).clamp(min=1)
    return (example_loss*class_weights[class_labels]).mean()

def evaluate_loss(loader):
    model.eval();total,count=0.0,0
    with torch.no_grad():
        for batch in loader:
            ids=batch["input_ids"].to(device)
            mask=batch["attention_mask"].to(device)
            labels=batch["labels"].to(device)
            classes=batch["class_label"].to(device)
            logits=model(input_ids=ids,attention_mask=mask).logits
            loss=weighted_generation_loss(logits,labels,classes)
            n=ids.size(0);total+=loss.item()*n;count+=n
    return total/max(count,1)

# ============================================================
# 10. CALCULATE HPV SCORES AND SELECT VALIDATION THRESHOLD
# ============================================================
yes_ids=tokenizer.encode(" Yes",add_special_tokens=False)
no_ids=tokenizer.encode(" No",add_special_tokens=False)
if len(yes_ids)!=1 or len(no_ids)!=1:
    raise ValueError("' Yes' and ' No' must each be a single GPT-2 token.")
YES_TOKEN_ID,NO_TOKEN_ID=yes_ids[0],no_ids[0]

def get_hpv_scores(prompts,batch_size=BATCH_SIZE):
    model.eval();scores=[]
    with torch.no_grad():
        for start in range(0,len(prompts),batch_size):
            batch_prompts=prompts[start:start+batch_size]
            encoded=tokenizer(batch_prompts,return_tensors="pt",padding=True,truncation=True,max_length=MAX_LENGTH).to(device)
            logits=model(input_ids=encoded["input_ids"],attention_mask=encoded["attention_mask"]).logits
            last_positions=encoded["attention_mask"].sum(dim=1)-1
            next_logits=logits[torch.arange(len(batch_prompts),device=device),last_positions]
            probs=torch.softmax(next_logits,dim=-1)
            yes_prob=probs[:,YES_TOKEN_ID];no_prob=probs[:,NO_TOKEN_ID]
            scores.extend((yes_prob/(yes_prob+no_prob).clamp(min=1e-12)).cpu().numpy())
    return np.asarray(scores,dtype=float)

def select_threshold(scores,labels):
    thresholds=np.unique(np.concatenate(([0.0],scores,[1.0])))
    results=[(float(t),balanced_accuracy_score(labels,(scores>=t).astype(int))) for t in thresholds]
    return max(results,key=lambda x:(x[1],-abs(x[0]-0.5)))

# ============================================================
# 11. TRAIN GPT-2 AND SAVE FOUR BEST VALIDATION CHECKPOINTS
# ============================================================
from pathlib import Path

OUT_DIR = Path("/content/drive/MyDrive/HECKTOR_GPT2_Generative_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_paths={"Validation_Loss":OUT_DIR/"best_model_validation_loss.pth","Validation_ROC_AUC":OUT_DIR/"best_model_validation_roc_auc.pth","Validation_PR_AUC":OUT_DIR/"best_model_validation_pr_auc.pth","Validation_Balanced_Accuracy":OUT_DIR/"best_model_validation_balanced_accuracy.pth"}
best={"Validation_Loss":{"value":float("inf"),"epoch":None,"threshold":None},"Validation_ROC_AUC":{"value":-float("inf"),"epoch":None,"threshold":None},"Validation_PR_AUC":{"value":-float("inf"),"epoch":None,"threshold":None},"Validation_Balanced_Accuracy":{"value":-float("inf"),"epoch":None,"threshold":None}}
history=[]

for epoch in range(1,EPOCHS+1):
    model.train();train_total,train_count=0.0,0
    for batch in train_loader:
        ids=batch["input_ids"].to(device);mask=batch["attention_mask"].to(device)
        labels=batch["labels"].to(device);classes=batch["class_label"].to(device)
        optimizer.zero_grad(set_to_none=True)
        logits=model(input_ids=ids,attention_mask=mask).logits
        loss=weighted_generation_loss(logits,labels,classes)
        loss.backward();optimizer.step();n=ids.size(0)
        train_total+=loss.item()*n;train_count+=n
    train_loss=train_total/max(train_count,1)
    val_loss=evaluate_loss(val_loader)
    val_scores=get_hpv_scores(val_texts)
    val_auc=roc_auc_score(y_val,val_scores)
    val_ap=average_precision_score(y_val,val_scores)
    val_threshold,val_ba=select_threshold(val_scores,y_val)
    history.append({"Epoch":epoch,"Train_Loss":train_loss,"Validation_Loss":val_loss,"Validation_ROC_AUC":val_auc,"Validation_PR_AUC":val_ap,"Validation_Balanced_Accuracy":val_ba,"Validation_Threshold":val_threshold})
    values={"Validation_Loss":val_loss,"Validation_ROC_AUC":val_auc,"Validation_PR_AUC":val_ap,"Validation_Balanced_Accuracy":val_ba}
    for metric,value in values.items():
        better=value<best[metric]["value"] if metric=="Validation_Loss" else value>best[metric]["value"]
        if better:
            best[metric]={"value":value,"epoch":epoch,"threshold":val_threshold}
            torch.save(model.state_dict(),checkpoint_paths[metric])
    if epoch%50==0:
        print(f"Epoch {epoch}/{EPOCHS} | Train Loss={train_loss:.4f} | Val Loss={val_loss:.4f} | Val ROC-AUC={val_auc:.4f} | Val PR-AUC={val_ap:.4f} | Val BA={val_ba:.4f} | Threshold={val_threshold:.6f}")

pd.DataFrame(history).to_csv(OUT_DIR/"training_history.csv",index=False)
best_summary=pd.DataFrame([{"Checkpoint_Selected_By":metric,"Best_Epoch":value["epoch"],"Best_Validation_Metric":value["value"],"Threshold_Selected_On_Validation":value["threshold"],"Checkpoint_Path":str(checkpoint_paths[metric])} for metric,value in best.items()])
best_summary.to_csv(OUT_DIR/"best_validation_checkpoints.csv",index=False)
print("\nBEST VALIDATION CHECKPOINTS\n",best_summary.to_string(index=False))

# ============================================================
# 12. EVALUATE FOUR CHECKPOINTS ON HELD-OUT TEST SET
# ============================================================
test_results=[];test_predictions={}
for metric,path in checkpoint_paths.items():
    print(f"\n===== CHECKPOINT: {metric} =====")
    model.load_state_dict(torch.load(path,map_location=device))
    scores=get_hpv_scores(test_texts)
    threshold=best[metric]["threshold"]
    pred=(scores>=threshold).astype(int)
    auc=roc_auc_score(y_test,scores)
    ap=average_precision_score(y_test,scores)
    ba=balanced_accuracy_score(y_test,pred)
    f1=f1_score(y_test,pred,zero_division=0)
    cm=confusion_matrix(y_test,pred,labels=[0,1])
    print("Best epoch:",best[metric]["epoch"])
    print(f"Validation threshold: {threshold:.6f}")
    print(classification_report(y_test,pred,labels=[0,1],target_names=["HPV Negative","HPV Positive"],digits=4,zero_division=0))
    print(f"Test ROC-AUC: {auc:.4f} | Test PR-AUC (AP): {ap:.4f} | Test Balanced Accuracy: {ba:.4f} | F1: {f1:.4f}")
    print("Confusion matrix [[TN, FP], [FN, TP]]:\n",cm)
    test_results.append({"Dataset":"HECKTOR","Model":"GPT-2 Generative","Checkpoint_Selected_By":metric,"Best_Epoch":best[metric]["epoch"],"Best_Validation_Metric":best[metric]["value"],"Threshold_Selected_On_Validation":threshold,"Test_ROC_AUC":auc,"Test_PR_AUC_Average_Precision":ap,"Test_Balanced_Accuracy":ba,"Test_F1":f1,"TN":int(cm[0,0]),"FP":int(cm[0,1]),"FN":int(cm[1,0]),"TP":int(cm[1,1])})
    test_predictions[metric]={"scores":scores,"predictions":pred,"labels":y_test,"threshold":threshold}
    pd.DataFrame({"True_Label":y_test,"HPV_Positive_Score":scores,"Predicted_Label":pred}).to_csv(OUT_DIR/f"test_predictions_{metric.lower()}.csv",index=False)

results_df=pd.DataFrame(test_results)
results_df.to_csv(OUT_DIR/"test_metrics_by_checkpoint.csv",index=False)
print("\nTEST RESULTS BY CHECKPOINT\n",results_df.to_string(index=False))

# ============================================================
# 13. SAVE ROC AND PRECISION-RECALL CURVES FOR ALL CHECKPOINTS
# ============================================================
for metric,result in test_predictions.items():
    labels=result["labels"];scores=result["scores"]
    auc=roc_auc_score(labels,scores);ap=average_precision_score(labels,scores)
    fpr,tpr,roc_thresholds=roc_curve(labels,scores)
    precision,recall,_=precision_recall_curve(labels,scores)
    safe_name=metric.lower()
    pd.DataFrame({"FPR":fpr,"TPR":tpr,"Threshold":roc_thresholds}).to_csv(OUT_DIR/f"roc_coordinates_{safe_name}.csv",index=False)
    pd.DataFrame({"Precision":precision,"Recall":recall}).to_csv(OUT_DIR/f"precision_recall_coordinates_{safe_name}.csv",index=False)
    plt.figure(figsize=(7,6))
    plt.plot(fpr,tpr,linewidth=2,label=f"ROC-AUC = {auc:.4f}")
    plt.plot([0,1],[0,1],"--",linewidth=1.5,label="Random")
    plt.xlabel("False Positive Rate");plt.ylabel("True Positive Rate")
    plt.title(f"HECKTOR GPT-2 Generative ROC: {metric}")
    plt.legend(loc="lower right");plt.grid(alpha=0.3);plt.tight_layout()
    plt.savefig(OUT_DIR/f"roc_curve_{safe_name}.png",dpi=300,bbox_inches="tight")
    plt.close()
    plt.figure(figsize=(7,6))
    plt.plot(recall,precision,linewidth=2,label=f"PR-AUC (AP) = {ap:.4f}")
    plt.xlabel("Recall");plt.ylabel("Precision")
    plt.title(f"HECKTOR GPT-2 Generative Precision-Recall: {metric}")
    plt.legend(loc="lower left");plt.grid(alpha=0.3);plt.tight_layout()
    plt.savefig(OUT_DIR/f"precision_recall_curve_{safe_name}.png",dpi=300,bbox_inches="tight")
    plt.close()

# ============================================================
# 14. SAVE SETTINGS AND MODEL EVALUATION OUTPUTS
# ============================================================
settings={"Dataset":"HECKTOR","Model":MODEL_NAME,"Features":", ".join(features),"Number_of_Features":len(features),"Target":"HPV Status","Excluded_Features":"Relapse, RFS, Treatment, PatientID, CenterID, Task 1, Task 2, Task 3","Learning_Rate":LEARNING_RATE,"Weight_Decay":WEIGHT_DECAY,"Optimizer":"AdamW","Epochs":EPOCHS,"Batch_Size":BATCH_SIZE,"Max_Length":MAX_LENGTH,"Max_Observed_Training_Prompt_And_Target_Tokens":max(TRAIN_REQUIRED_LENGTHS),"Random_Seed":SEED,"Checkpoint_Selection":"Best validation loss, ROC-AUC, PR-AUC and balanced accuracy","Threshold_Selection":"Maximise validation balanced accuracy; ties closest to 0.5","F1":"Reported at the validation-selected threshold","Curves":"Test ROC and precision-recall curves saved for each checkpoint"}
pd.DataFrame([settings]).to_csv(OUT_DIR/"settings.csv",index=False)
print("\nFeatures used:",features)
print("All HECKTOR results saved to:",OUT_DIR.resolve())

# ============================================================
# 15. APPEND HECKTOR RESULTS TO MODEL COMPARISON CSV
# ============================================================
comparison_file=Path("model_roc_pr_auc_comparison.csv")
hecktor_results=results_df[["Dataset","Model","Checkpoint_Selected_By","Best_Epoch","Test_ROC_AUC","Test_PR_AUC_Average_Precision","Test_Balanced_Accuracy","Test_F1"]].copy()
hecktor_results=hecktor_results.rename(columns={"Test_PR_AUC_Average_Precision":"Test_PR_AUC"})
if os.path.exists(comparison_file):
    existing=pd.read_csv(comparison_file)
    for col in hecktor_results.columns:
        if col not in existing.columns:existing[col]=np.nan
    for col in existing.columns:
        if col not in hecktor_results.columns:hecktor_results[col]=np.nan
    combined=pd.concat([existing,hecktor_results[existing.columns]],ignore_index=True)
    combined=combined.drop_duplicates(subset=["Dataset","Model","Checkpoint_Selected_By"],keep="last")
else:
    combined=hecktor_results
combined.to_csv(comparison_file,index=False)
print("AUC comparison saved to:",comparison_file)
print(combined.to_string(index=False))

Device: cuda


Downloading...
From: https://drive.google.com/uc?id=1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv
To: /content/HPV2025.xlsx
100%|██████████| 66.0k/66.0k [00:00<00:00, 89.4MB/s]


Patients: 588
HPV class distribution:
 HPV Status
0     58
1    530
Name: count, dtype: int64
Features: ['Age', 'Gender', 'Tobacco Consumption', 'Alcohol Consumption', 'Performance Status', 'T-stage', 'N-stage', 'M-stage']
Train: 381 classes: [ 37 344]
Validation: 89 classes: [ 9 80]
Test: 118 classes: [ 12 106]
Example clinical narrative:
 Q: Predict this patient's Human Papillomavirus (HPV) status using the clinical information available. Age at diagnosis: 66. Gender: male. Tobacco consumption: current smoker. Alcohol consumption: consumes alcohol. Performance status: 0. Tumour stage: T1. Nodal stage: N2. Metastatic stage: M0. Is this patient HPV positive? Answer with only Yes or No.
A: 
Target: Yes


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT-2 hidden size: 768 | Layers: 12 | Heads: 12
Class weights: [5.1486487  0.55377907]


/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 50/500 | Train Loss=0.1478 | Val Loss=1.6265 | Val ROC-AUC=0.4125 | Val PR-AUC=0.8969 | Val BA=0.5938 | Threshold=0.999894
Epoch 100/500 | Train Loss=0.0764 | Val Loss=5.3100 | Val ROC-AUC=0.4542 | Val PR-AUC=0.8898 | Val BA=0.5118 | Threshold=0.591997
Epoch 150/500 | Train Loss=0.0530 | Val Loss=4.9639 | Val ROC-AUC=0.3799 | Val PR-AUC=0.8702 | Val BA=0.5243 | Threshold=0.780215
Epoch 200/500 | Train Loss=0.0611 | Val Loss=3.6993 | Val ROC-AUC=0.5674 | Val PR-AUC=0.9164 | Val BA=0.5958 | Threshold=1.000000
Epoch 250/500 | Train Loss=0.0319 | Val Loss=5.6250 | Val ROC-AUC=0.5472 | Val PR-AUC=0.9079 | Val BA=0.5799 | Threshold=0.993615
Epoch 300/500 | Train Loss=0.0316 | Val Loss=5.4022 | Val ROC-AUC=0.5736 | Val PR-AUC=0.9145 | Val BA=0.5917 | Threshold=1.000000
Epoch 350/500 | Train Loss=0.0948 | Val Loss=1.9186 | Val ROC-AUC=0.6181 | Val PR-AUC=0.9274 | Val BA=0.6340 | Threshold=0.995753
Epoch 400/500 | Train Loss=0.0131 | Val Loss=4.2127 | Val ROC-AUC=0.6160 | Val PR-AUC=0.922

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score
)

from statsmodels.stats.proportion import proportion_confint

# Output folder
folder = Path("/content/drive/MyDrive/HECKTOR_GPT2_Generative_results")

# Load selected checkpoint predictions
file = folder / "test_predictions_validation_roc_auc.csv"

df = pd.read_csv(file)

# Extract actual labels, predicted probabilities and class predictions
y_true = df["True_Label"].to_numpy()
y_prob = df["HPV_Positive_Score"].to_numpy()
y_pred = df["Predicted_Label"].to_numpy()

# Calculate point estimates
roc = roc_auc_score(y_true, y_prob)
pr = average_precision_score(y_true, y_prob)
ba = balanced_accuracy_score(y_true, y_pred)

# Bootstrap confidence intervals
rng = np.random.default_rng(42)

boot_roc = []
boot_pr = []
boot_ba = []

for _ in range(2000):

    idx = rng.integers(0, len(y_true), len(y_true))

    yt = y_true[idx]
    yp = y_prob[idx]
    yh = y_pred[idx]

    if len(np.unique(yt)) < 2:
        continue

    boot_roc.append(roc_auc_score(yt, yp))
    boot_pr.append(average_precision_score(yt, yp))
    boot_ba.append(balanced_accuracy_score(yt, yh))

def ci(values):
    return np.percentile(values, [2.5, 97.5])

# Confusion matrix components
tn = np.sum((y_true == 0) & (y_pred == 0))
fp = np.sum((y_true == 0) & (y_pred == 1))
fn = np.sum((y_true == 1) & (y_pred == 0))
tp = np.sum((y_true == 1) & (y_pred == 1))

# Wilson confidence intervals
sens_ci = proportion_confint(tp, tp + fn, method="wilson")
spec_ci = proportion_confint(tn, tn + fp, method="wilson")

# Compile results
results = pd.DataFrame([
    ["ROC-AUC", roc, *ci(boot_roc)],
    ["PR-AUC (Average Precision)", pr, *ci(boot_pr)],
    ["Balanced Accuracy", ba, *ci(boot_ba)],
    ["Sensitivity (HPV positive)", tp/(tp+fn), *sens_ci],
    ["Specificity (HPV negative)", tn/(tn+fp), *spec_ci]
], columns=[
    "Metric", "Estimate", "CI_lower", "CI_upper"
])

# Save results
output = folder / "HECKTOR_GPT2_Generative_confidence_intervals.csv"

results.to_csv(output, index=False)

print(results.round(4).to_string(index=False))
print("\nConfidence intervals saved to:", output)

                    Metric  Estimate  CI_lower  CI_upper
                   ROC-AUC    0.7472    0.6091    0.8727
PR-AUC (Average Precision)    0.9651    0.9328    0.9889
         Balanced Accuracy    0.7186    0.5838    0.8208
Sensitivity (HPV positive)    0.6038    0.5086    0.6917
Specificity (HPV negative)    0.8333    0.5520    0.9530

Confidence intervals saved to: /content/drive/MyDrive/HECKTOR_GPT2_Generative_results/HECKTOR_GPT2_Generative_confidence_intervals.csv
